In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import torch

d:\fine-tune-text-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# # Data
# spam_examples = [
#     "URGENT! You've won $1000000! Click here NOW!",
#     "FREE VIAGRA! No prescription needed! Call 555-SPAM",
#     "Congratulations! You're our 1000th visitor! Claim your prize!",
#     "Make $5000/week working from home! No experience required!",
#     "STOP! Don't pay your credit card bill until you read this!"
# ]

# ham_examples = [
#     "Hey, are we still meeting for lunch tomorrow?",
#     "Your package has been delivered to your front door.",
#     "Meeting rescheduled to 3 PM. See you in conference room B.",
#     "Thanks for the great presentation today!",
#     "Your subscription renewal is due next month."
# ]

# data = {
#     'text': spam_examples + ham_examples,
#     'labels': [1]*len(spam_examples) + [0]*len(ham_examples)
# }
# dataset = Dataset.from_dict(data)

In [4]:
# Typical spam vs ham examples in Thai
spam_examples = [
    "ด่วน! คุณชนะรางวัล 1,000,000 บาท! คลิกที่นี่เลย!",
    "ฟรี! ยาเพิ่มสมรรถภาพ! ไม่ต้องใบสั่งแพทย์! โทร 555-SPAM",
    "ยินดีด้วย! คุณเป็นผู้เข้าชมคนที่ 1000! รับรางวัลของคุณ!",
    "หารายได้ 5,000 บาท/สัปดาห์ ทำงานที่บ้าน! ไม่ต้องมีประสบการณ์!",
    "หยุด! อย่าจ่ายบัตรเครดิตจนกว่าคุณจะอ่านข้อความนี้!"
]

ham_examples = [
    "เฮ้ เรายังนัดกินข้าวเที่ยงพรุ่งนี้อยู่ใช่ไหม?",
    "พัสดุของคุณถูกส่งถึงหน้าประตูบ้านแล้ว",
    "ประชุมเลื่อนเป็นบ่าย 3 โมง พบกันที่ห้องประชุม B",
    "ขอบคุณสำหรับการนำเสนอที่ยอดเยี่ยมวันนี้!",
    "การต่ออายุสมาชิกของคุณครบกำหนดเดือนหน้า"
]

data = {
    'text': spam_examples + ham_examples,
    'labels': [1]*len(spam_examples) + [0]*len(ham_examples)
}
dataset = Dataset.from_dict(data)

In [ ]:
# Load model
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# Tokenize
def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, padding=True)

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 10/10 [00:00<00:00, 2498.54 examples/s]


In [ ]:
# Add LoRA
# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=16,
#     target_modules=["q_lin", "v_lin"],  # DistilBERT attention layers
#     lora_dropout=0.1,
#     bias="none",
#     task_type="SEQ_CLS"
# )

lora_config = LoraConfig(
    r=32, 
    lora_alpha=64,
    target_modules=["q_lin", "v_lin", "k_lin", "out_lin"],
    lora_dropout=0.1
)


model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12462.65it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,179,648 || all params: 68,134,658 || trainable%: 1.7313


In [25]:
# Train
training_args = TrainingArguments(
    output_dir='./artifacts/spam_model_lora',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=6, training_loss=0.7144308090209961, metrics={'train_runtime': 2.5748, 'train_samples_per_second': 11.651, 'train_steps_per_second': 2.33, 'total_flos': 111637375920.0, 'train_loss': 0.7144308090209961, 'epoch': 3.0})

In [26]:
# Save
model.save_pretrained('./artifacts/spam_model_lora')
tokenizer.save_pretrained('./artifacts/spam_model_lora')

('./artifacts/spam_model_lora\\tokenizer_config.json',
 './artifacts/spam_model_lora\\tokenizer.json')

In [27]:
# Predict
# test_texts = ["WIN FREE MONEY NOW!!!", "Let's grab coffee tomorrow"]
test_texts = [
    "ชนะเงินฟรีตอนนี้!!!",
    "ไปดื่มกาแฟกันพรุ่งนี้นะ"
]

inputs = tokenizer(test_texts, truncation=True, padding=True, return_tensors="pt").to(model.device)
outputs = model(**inputs)
predictions = torch.argmax(outputs.logits, dim=-1)
for text, pred in zip(test_texts, predictions):
    print(f"{text}: {'SPAM' if pred == 1 else 'HAM'}")

ชนะเงินฟรีตอนนี้!!!: HAM
ไปดื่มกาแฟกันพรุ่งนี้นะ: HAM
